### Algoritmos hill_climbing y simulated_annealing

In [1]:
import random
import math


DEBUG = False

def debug (info):
    if (DEBUG):
       print (info)  

def hill_climbing(problem, max_iterations=1000):
    current_state = problem.state
    current_cost = problem.calculate_cost(current_state)
    debug (f"current_cost: {current_cost}")

    for _ in range(max_iterations):
        neighbors = problem.get_neighbors()
        debug (f"\t neighbors: {neighbors}")

        best_neighbor       = min(neighbors, key=lambda state: problem.calculate_cost(state))
        best_neighbor_cost  = problem.calculate_cost(best_neighbor)
        debug (f"\t\t best_neighbor: {best_neighbor}  -  {best_neighbor_cost}")

        if best_neighbor_cost >= current_cost: 
            return current_state
        
        problem.state = best_neighbor
        current_state = best_neighbor
        current_cost = best_neighbor_cost

   

def simulated_annealing(problem, max_iterations=10000000):
    current_state = problem.state
    current_cost  = problem.calculate_cost(current_state)
    debug(f"current_state, {current_state}: {current_cost}")
    
    current_temperature = max_iterations

    while (not problem.is_goal_state()) and (current_temperature > 0):
        neighbors = problem.get_neighbors()
        debug(f"\t Neighbors:, {neighbors}")

        next_state = random.choice(neighbors)
        next_cost = problem.calculate_cost(next_state)  # Calculando el costo del próximo estado
        debug(f"\t\t next_state  {next_state} ({next_cost})")

        diff_cost = next_cost - current_cost       
        move =  False
        
        if (diff_cost <= 0):
            move = True
        else:
            boltz = math.exp(-float(diff_cost) / current_temperature)
            randomvalue = random.random()
            debug(f"\t\t diff_cost  {diff_cost}")
            debug(f"\t\t randomvalue  {randomvalue}, boltz {boltz}")
            if  randomvalue <= boltz:
                move = True
        if move:
            problem.state = next_state
            current_state = next_state
            current_cost = next_cost
            debug(f"current_state, {problem.state}: {current_cost}")

        current_temperature-=1
    
    if (problem.is_goal_state()):
        print (f"Solución encontrada en: {current_temperature}/{max_iterations}")
    
    return current_state





# Clase Problem

In [2]:
import random

class Problem:
    def __init__(self, initial_state=None, goal_state=None):
        if initial_state is None:
            self.state = self.generate_random_state()
        else:
            self.state = initial_state  
        
        if goal_state is None:
            self.goal_state = self.generate_random_state()
        else:
            self.goal_state = goal_state  


    def display_state(self):
        raise NotImplementedError("Subclasses must implement display_state method")

    def generate_random_state(self):
        raise NotImplementedError("Subclasses must implement generate_random_state method")

    def calculate_cost(self, state):
        raise NotImplementedError("Subclasses must implement calculate_cost method")

    def get_neighbors(self):
        raise NotImplementedError("Subclasses must implement get_neighbors method")
  
    def is_goal_state(self):
        raise NotImplementedError("Subclasses must implement get_neighbors method")




# Problema puzzle 8

In [3]:
class Puzzle8(Problem):
    def display_state(self):
        for i in range(3):
            print(self.state[i * 3:i * 3 + 3])

    def generate_random_state(self):
        return random.sample(range(9), 9)  # Generate a random state as a list of numbers 0-8

    def calculate_cost(self, state):
        energy = 0
        for i in state:
            if state[i] != 0:  # Ignore the empty space
                energy += abs(state[i]-(i+1))
        return energy

    def get_neighbors(self):
     neighbors = []
     zero_index = self.state.index(0)
     row, col = divmod(zero_index, 3)  # Convertir el índice en fila y columna
     moves = [(1, 0), (-1, 0), (0, 1), (0, -1)]  # Movimientos: arriba, abajo, izquierda, derecha
     for move in moves:
        new_row, new_col = row + move[0], col + move[1]
        if 0 <= new_row < 3 and 0 <= new_col < 3:  # Verificar límites
            new_index = new_row * 3 + new_col
            new_state = list(self.state)
            new_state[zero_index], new_state[new_index] = new_state[new_index], new_state[zero_index]
            neighbors.append(new_state)     
     return neighbors

    def is_goal_state(self):
        return self.state == self.goal_state


# Ejecución Puzzle 8 por  Hill Climbing

In [4]:
# Ejemplo Transparencias Finaliza
initial_state = [1, 2, 3, 4, 8, 5, 7, 0, 6]
goal_state    = [1, 2, 3, 4, 5, 6, 7, 8, 0]

problem = Puzzle8(initial_state,goal_state)
solution = hill_climbing(problem)
print("Estado final encontrado por Hill Climbing:")
problem.display_state()
print("¿El estado devuelto es el objetivo?", problem.is_goal_state())

print("----------------")
# Example usage
initial_state = [1, 2, 3, 4, 0, 5, 6, 8, 7]
goal_state    = [1, 2, 3, 4, 5, 6, 7, 8, 0]

problem = Puzzle8(initial_state,goal_state)
solution = hill_climbing(problem)
print("Estado final encontrado por Hill Climbing:")
problem.display_state()
print("¿El estado devuelto es el objetivo?", problem.is_goal_state())

Estado final encontrado por Hill Climbing:
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]
¿El estado devuelto es el objetivo? True
----------------
Estado final encontrado por Hill Climbing:
[1, 2, 3]
[4, 5, 7]
[6, 8, 0]
¿El estado devuelto es el objetivo? False


# Ejecución Puzzle 8 por Simulated Annealing

In [5]:
print("----------------")
initial_state = [4, 1, 7, 6, 0, 2, 3, 5, 8]
goal_state    = [1, 2, 3, 4, 5, 6, 7, 8, 0]

problem = Puzzle8(initial_state,goal_state)
solution = simulated_annealing(problem)
print("Estado final encontrado por Simulated Annealing:")
problem.display_state()
print("¿El estado devuelto es el objetivo?", problem.is_goal_state())

----------------
Solución encontrada en: 9491212/10000000
Estado final encontrado por Simulated Annealing:
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]
¿El estado devuelto es el objetivo? True


# Problema nreinas

In [6]:
class nreinas(Problem):
    def __init__(self, queen_count=8):
        self.queen_count = queen_count
        self.state       = self.generate_random_state()

    def calculate_cost(self,state):
        threat = 0

        for queen in range(0, self.queen_count):
            for next_queen in range(queen+1, self.queen_count):
                if state[queen] == state[next_queen] or abs(queen - next_queen) == abs(state[queen] - state[next_queen]):
                    threat += 1
        return threat
    
    def display_state_plain(self):
       print(self.state)
        
    def display_state_duplas(self):
       board_string = ""
       for row, col in enumerate(self.state):
            board_string += "(%s, %s)\n" % (row, col)
       print(board_string)

    def display_state(self):
        max_y = len(self.state)
        max_x = max_y
        
        matriz = [[' ' for _ in range(max_x)] for _ in range(max_y)]
        
        for y, x in enumerate(self.state):
            matriz[y][x] = 'X'
        
        matriz_str = ''
        for i in range(max_y - 1, -1, -1):
            matriz_str += '--+' + '-' * (max_x * 4 - 1) + '+\n'
            matriz_str += f'{i} | {" | ".join(str(x) for x in matriz[i])} |\n'
            #if i > 0:
           
        matriz_str += '--+' + '-' * (max_x * 4 - 1) + '+\n'
        matriz_str += '    ' + '   '.join(str(i) for i in range(max_x))
        
        print(matriz_str)
        
     
    def generate_random_state(self):
        return random.sample(range(9), 9)  


    def get_neighbors(self):
      neighbors = []
      queens = [-1 for i in range(0, self.queen_count)]

      for i in range(0, self.queen_count):
            queens[i] = random.randint(0, self.queen_count - 1)
      neighbors.append(queens)     
      return neighbors
     
    def is_goal_state(self):
        return (self.calculate_cost(self.state) ==0 )

# Ejecución n-reinas con Simulated Annealing

In [7]:
print("----------------")


problem = nreinas()
solution = simulated_annealing(problem)
print("Estado final encontrado por Simulated Annealing:")
problem.display_state()
print("¿El estado devuelto es el objetivo?", problem.is_goal_state())

----------------
Solución encontrada en: 9996772/10000000
Estado final encontrado por Simulated Annealing:
--+-------------------------------+
7 |   |   | X |   |   |   |   |   |
--+-------------------------------+
6 |   |   |   |   |   |   |   | X |
--+-------------------------------+
5 |   |   |   | X |   |   |   |   |
--+-------------------------------+
4 |   |   |   |   |   |   | X |   |
--+-------------------------------+
3 | X |   |   |   |   |   |   |   |
--+-------------------------------+
2 |   |   |   |   |   | X |   |   |
--+-------------------------------+
1 |   | X |   |   |   |   |   |   |
--+-------------------------------+
0 |   |   |   |   | X |   |   |   |
--+-------------------------------+
    0   1   2   3   4   5   6   7
¿El estado devuelto es el objetivo? True


## Algoritmo Genético: Ejemplo buscar una cadena objetivo

- Modifica el tamaño de la población. ¿Cuál es el tamaño de la población óptima?
- La probabilidad de mutación. ¿Es importante la mutación?

In [8]:
import random
import string

# Cadena de texto objetivo que queremos encontrar
objetivo = "¿quien ganara la eurocopa?"

# Tamaño de la población
tamano_poblacion = 1000

# Probabilidad de mutación
probabilidad_mutacion = 0.01

alfabeto = string.ascii_lowercase + ' '+ '?'+'¿'

def generar_cromosoma(longitud):
    cromosoma = ''
    for _ in range(longitud):
        cromosoma += random.choice(alfabeto)
    return cromosoma

def calcular_fitness(cromosoma):
    fitness = 0;
    for caracter_cromosoma, caracter_objetivo in zip(cromosoma, objetivo):
        if (caracter_cromosoma == caracter_objetivo):
            fitness +=1
    return fitness

def mutar(cromosoma):
    cromosoma_mutado = ''
    for caracter in cromosoma:
        if random.random() < probabilidad_mutacion:
            cromosoma_mutado += random.choice(alfabeto)
        else:
            cromosoma_mutado += caracter
    return cromosoma_mutado
    
def seleccion_padres(poblacion):
    return random.choices(poblacion, weights=[calcular_fitness(cromosoma) for cromosoma in poblacion], k=2)

def cruzar(padre1, padre2):
    punto_cruce = random.randint(0, len(padre1) - 1)
    hijo = padre1[:punto_cruce] + padre2[punto_cruce:]
    return hijo

# Crear una población inicial aleatoria
poblacion = [generar_cromosoma(len(objetivo)) for _ in range(tamano_poblacion)]

generacion = 0
while True:
    generacion += 1

    # Calcular el fitness de cada individuo en la población
    fitness_poblacion = [calcular_fitness(cromosoma) for cromosoma in poblacion]

    # Comprobar si alguno de los individuos ha alcanzado la cadena objetivo
    if max(fitness_poblacion) == len(objetivo):
        indice = fitness_poblacion.index(len(objetivo))
        print(f"Generación {generacion}: ¡Se encontró la cadena '{poblacion[indice]}'!")
        break

    # Seleccionar los padres para la reproducción
    padres = seleccion_padres(poblacion)

    # Cruzar los padres para producir descendencia
    descendencia = cruzar(padres[0], padres[1])

    # Aplicar mutación a la descendencia
    descendencia_mutada = mutar(descendencia)

    # Reemplazar al individuo menos apto en la población con la descendencia mutada
    indice_menos_apto = fitness_poblacion.index(min(fitness_poblacion))
    poblacion[indice_menos_apto] = descendencia_mutada

    # Imprimir el mejor individuo de cada generación
    mejor_fitness = max(fitness_poblacion)
    mejor_individuo = poblacion[fitness_poblacion.index(mejor_fitness)]
    print(f"Generación {generacion}: {mejor_individuo} (Fitness: {mejor_fitness})")

Generación 1: ¿sc ir patmot dg rx?emiqql (Fitness: 5)
Generación 2: ¿sc ir patmot dg rx?emiqql (Fitness: 5)
Generación 3: ¿sc ir patmot dg rx?emiqql (Fitness: 5)
Generación 4: ¿sc ir patmot dg rx?emiqql (Fitness: 5)
Generación 5: ¿sc ir patmot dg rx?emiqql (Fitness: 5)
Generación 6: ¿sc ir patmot dg rx?emiqql (Fitness: 5)
Generación 7: s¿sibfkzqdpmaqldremolyrpgu (Fitness: 5)
Generación 8: s¿sibfkzqdpmaqldremolyrpgu (Fitness: 5)
Generación 9: s¿sibfkzqdpmaqldremolyrpgu (Fitness: 5)
Generación 10: s¿sibfkzqdpmaqldremolyrpgu (Fitness: 5)
Generación 11: s¿sibfkzqdpmaqldremolyrpgu (Fitness: 5)
Generación 12: s¿sibfkzqdpmaqldremolyrpgu (Fitness: 5)
Generación 13: s¿sibfkzqdpmaqldremolyrpgu (Fitness: 5)
Generación 14: s¿sibfkzqdpmaqldremolyrpgu (Fitness: 5)
Generación 15: s¿sibfkzqdpmaqldremolyrpgu (Fitness: 5)
Generación 16: wsa¿e¿lgsvrrh ?uleda¿qwbd? (Fitness: 6)
Generación 17: wsa¿e¿lgsvrrh ?uleda¿qwbd? (Fitness: 6)
Generación 18: wsa¿e¿lgsvrrh ?uleda¿qwbd? (Fitness: 6)
Generación 19: wsa¿